# 🔧 SpamShield — Feature Extraction
**Project:** Deteksi Spam Promosi Judi Online pada Komentar YouTube Indonesia  
**Authors:** Ricky (2702243016), Elmer Williams, Nicholas Sinclair  

---

## Scope & Tujuan

Notebook ini **hanya** bertanggung jawab untuk:
1. Mengekstrak **12 fitur handcrafted** dari teks komentar **ASLI** (sebelum normalisasi)
2. Menjalankan **normalisasi Unicode** dan menyimpan hasilnya sebagai kolom terpisah
3. Menyimpan dataset dengan fitur baru sebagai file CSV

**Yang TIDAK ada di notebook ini (ada di experiment notebook):**
- StandardScaler / normalisasi numerik
- train_test_split
- Dataset class / DataLoader
- Model definitions
- Training loop

---

## Pipeline Position
```
raw_75k.csv (dari scraper)
     ↓
[NOTEBOOK INI] feature_extraction.ipynb
     ↓
dataset_spamshield_features.csv  ← OUTPUT
     ↓
spamshield_experiments.ipynb (split + scaler + training)
```

---

## 12 Fitur Handcrafted (Sesuai Skripsi Section 1.4 & 3.3.2)

| # | Nama Fitur | Deskripsi | Kelompok |
|---|---|---|---|
| 1 | `text_length` | Panjang total karakter teks | Surface |
| 2 | `word_count` | Jumlah kata dalam komentar | Surface |
| 3 | `emoji_ratio` | Rasio kemunculan emoji | Surface |
| 4 | `uppercase_ratio` | Rasio penggunaan huruf kapital | Surface |
| 5 | `special_char_ratio` | Rasio karakter khusus | Surface |
| 6 | `repeat_char_ratio` | Rasio pengulangan karakter berurutan | Surface |
| 7 | `unicode_weird_ratio` | Rasio karakter Unicode non-standar (cp > 10000) | Obfuskasi |
| 8 | `invisible_char_ratio` | Rasio karakter kontrol/tersembunyi (Cf, Cc) | Obfuskasi |
| 9 | `masked_keyword_ratio` | Rasio kata kunci spam yang tersamarkan | Obfuskasi |
| 10 | `stretched_word_ratio` | Rasio kata dengan pemanjangan tidak wajar | Obfuskasi |
| 11 | `slang_ratio` | Rasio penggunaan kata tidak baku | Linguistik |
| 12 | `slang_distortion_ratio` | Rasio distorsi bentuk kata tidak baku | Linguistik |

> ⚠️ **PENTING:** Semua fitur diekstrak dari `comment_text` ASLI (sebelum normalisasi).  
> Normalisasi Unicode diterapkan setelahnya dan disimpan ke kolom `comment_text_normalized`.

## Cell 1 — Imports

In [1]:
import re
import json
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path

print("Libraries imported.")

Libraries imported.


## Cell 2 — Config & Paths

In [2]:
# ============================================================
# ⚠️  SESUAIKAN PATH DI SINI
# ============================================================

# Dataset sudah tersplit — proses masing-masing file
SPLIT_FILES = {
    "train": "./train.csv",
    "val"  : "./val.csv",
    "test" : "./test.csv",
}

# Output: nama file ditambah suffix _features.csv
# Contoh: train.csv → train_features.csv

SLANG_DICT_PATH = "./slangwords.txt"  # path ke file slang dict (format JSON)

TEXT_COL  = "comment_text"
LABEL_COL = "final_label"   # string: "SPAM" / "NOT_SPAM"

# Mapping label string → numerik
LABEL_MAP = {"SPAM": 1, "NOT_SPAM": 0}

print("Split files:")
for split, path in SPLIT_FILES.items():
    import os
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"  {split:<8}: {path}  {status}")

Split files:
  train   : ./train.csv  ✅
  val     : ./val.csv  ✅
  test    : ./test.csv  ✅


## Cell 3 — Load Dataset

In [3]:
import os

# Validasi semua split files ada
errors = []
for split, path in SPLIT_FILES.items():
    if not os.path.exists(path):
        errors.append(f"  ❌ {split}: '{path}' tidak ditemukan")

if errors:
    raise FileNotFoundError("\n".join(["File tidak ditemukan:"] + errors))

# Cek sample dari train.csv
df_sample = pd.read_csv(SPLIT_FILES["train"], nrows=3)
print("Sample kolom dari train.csv:")
print(f"  Columns : {list(df_sample.columns)}")
print(f"  Rows    : {sum(len(pd.read_csv(p)) for p in SPLIT_FILES.values()):,} total")
print()
for split, path in SPLIT_FILES.items():
    df = pd.read_csv(path)
    spam_pct = (df[LABEL_COL] == "SPAM").mean() * 100
    print(f"  {split:<8}: {len(df):,} rows | SPAM={spam_pct:.1f}%")

Sample kolom dari train.csv:
  Columns : ['comment_id', 'comment_text', 'comment_published_at', 'comment_updated_at', 'author_display_name', 'like_count', 'reply_count', 'channel_subscriber_count', 'channel_video_count', 'channel_view_count', 'channel_created_at', 'channel_description_length', 'video_comment_count', 'text_length', 'word_count', 'emoji_ratio', 'uppercase_ratio', 'special_char_ratio', 'repeat_char_ratio', 'unicode_weird_ratio', 'invisible_char_ratio', 'masked_keyword_ratio', 'stretched_word_ratio', 'slang_ratio', 'slang_distortion_ratio', 'any_obf', 'llm_label', 'llm_confidence', 'llm_reason', 'llm_source', 'final_label', 'label_source']
  Rows    : 68,765 total

  train   : 55,012 rows | SPAM=6.0%
  val     : 6,876 rows | SPAM=6.0%
  test    : 6,877 rows | SPAM=6.0%


---
## Bagian 1 — Normalisasi Unicode (Section 3.3.2 Skripsi)

Fungsi normalisasi diterapkan ke `comment_text` untuk menghasilkan  
`comment_text_normalized` yang akan digunakan oleh IndoRoBERTa.

> ⚠️ Normalisasi ini dijalankan **SETELAH** ekstraksi fitur.  
> Urutan yang benar: ekstrak fitur dulu → baru normalisasi.

### Cell 4 — Fungsi Normalisasi Unicode

In [4]:
# Zero-width characters yang perlu dihapus
ZW_CHARS = [
    '\u200b',  # Zero-Width Space
    '\u200c',  # Zero-Width Non-Joiner
    '\u200d',  # Zero-Width Joiner
    '\u2060',  # Word Joiner
    '\ufeff',  # BOM
    '\u200e',  # Left-to-Right Mark
    '\u200f',  # Right-to-Left Mark
]

# ──────────────────────────────────────────────────────────────────────────
# Leet-speak normalization — SESUAI THESIS SECTION 3.3.2
# @ or 4 → a | 3 → e | 1 → i | 5 or $ → s | 7 → t
# ──────────────────────────────────────────────────────────────────────────
LEET_NORMALIZATION = str.maketrans({
    '4': 'a', '@': 'a',
    '3': 'e',
    '1': 'i',
    # NOTE: '0' → 'o' TIDAK ada di thesis, sengaja tidak dimasukkan
    '$': 's', '5': 's',
    '7': 't',
})


def normalize_unicode(text: str) -> str:
    """
    Normalisasi teks komentar sesuai Skripsi Section 3.3.2.

    Langkah:
    1. Unicode NFC normalization
    2. Hapus zero-width characters
    3. Leet-speak normalization (@ 4 → a, 3 → e, 1 → i, 5 $ → s, 7 → t)
    4. Reduksi pengulangan karakter berlebihan (gaaacooorrr → gacor)
    """
    if pd.isna(text):
        return ""
    text = str(text)

    # Step 1: NFC normalization
    text = unicodedata.normalize("NFC", text)

    # Step 2: Hapus zero-width characters
    for zw in ZW_CHARS:
        text = text.replace(zw, "")

    # Step 3: Leet-speak normalization
    text = text.translate(LEET_NORMALIZATION)

    # Step 4: Reduksi pengulangan karakter (>= 3 berturut → 1)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    return text


# Quick test
samples = [
    "g4c0r b@nget slot 77",
    "ⓐⓜⓑⓘⓛ④ⓓ daftar sekarang",
    "gaaacooorrr maxwin",
    "main\u200bdi\u200bsini",
]
print("Normalization test:")
for s in samples:
    print(f"  Original : {repr(s)}")
    print(f"  Normal   : {repr(normalize_unicode(s))}")
    print()

Normalization test:
  Original : 'g4c0r b@nget slot 77'
  Normal   : 'gac0r banget slot tt'

  Original : 'ⓐⓜⓑⓘⓛ④ⓓ daftar sekarang'
  Normal   : 'ⓐⓜⓑⓘⓛ④ⓓ daftar sekarang'

  Original : 'gaaacooorrr maxwin'
  Normal   : 'gacor maxwin'

  Original : 'main\u200bdi\u200bsini'
  Normal   : 'maindisini'



---
## Bagian 2 — Feature Extraction (Section 3.3.2 & 3.4.2 Skripsi)

> ⚠️ **PENTING:** Semua fungsi ekstraksi fitur di bawah ini  
> dipanggil pada `comment_text` **ASLI** (sebelum normalisasi).  
> Ini sesuai thesis yang menyatakan fitur obfuskasi harus diambil  
> dari teks mentah agar informasi manipulasi karakter tidak hilang.

### Cell 5 — Helper Functions

In [5]:
def safe_ratio(count: int, length: int, min_len: int = 20) -> float:
    """
    Hitung rasio dengan floor pada panjang minimum.
    min_len=20 mencegah rasio ekstrem pada komentar sangat pendek.
    """
    effective_len = max(length, min_len)
    return count / effective_len


def clip(x: float, max_val: float = 1.0) -> float:
    """Clip nilai ke [0, max_val]."""
    return min(max(x, 0.0), max_val)


def tokenize(text: str) -> list:
    """Tokenisasi sederhana: lowercase + ekstrak word tokens."""
    if pd.isna(text):
        return []
    text = str(text).lower()
    return re.findall(r'\b\w+\b', text)


print("Helper functions defined: safe_ratio, clip, tokenize")

Helper functions defined: safe_ratio, clip, tokenize


### Cell 6 — Load Slang Dictionary

In [7]:
def load_slang_dict(path: str) -> dict:
    """
    Load kamus slang dari file JSON.
    Format: {"kata_slang": "kata_baku", ...}
    """
    with open(path, 'r', encoding='utf-8') as f:
        slang_dict = json.load(f)
    return {k.lower(): v.lower() for k, v in slang_dict.items()}


SLANG_DICT = load_slang_dict(SLANG_DICT_PATH)
print(f"Slang dict loaded: {len(SLANG_DICT):,} entries")
print(f"Sample: {list(SLANG_DICT.items())[:5]}")

Slang dict loaded: 4,317 entries
Sample: [('woww', 'wow'), ('aminn', 'amin'), ('met', 'selamat'), ('netaas', 'menetas'), ('keberpa', 'keberapa')]


### Cell 7 — Surface Features (Fitur 1–6)

In [8]:
# Emoji pattern — 4 blok Unicode utama
EMOJI_PATTERN = re.compile(
    "[\U0001F600-\U0001F64F"   # emoticons
    "\U0001F300-\U0001F5FF"    # misc symbols & pictographs
    "\U0001F680-\U0001F6FF"    # transport & map
    "\U0001F1E0-\U0001F1FF"    # flags
    "\U00002600-\U000027BF"    # misc symbols
    "\U0001F900-\U0001F9FF"    # supplemental symbols
    "]+",
    flags=re.UNICODE
)


def text_surface_features(text: str) -> dict:
    """
    Ekstrak 6 fitur surface dari teks ASLI.

    Returns:
        text_length        : panjang total karakter
        word_count         : jumlah kata
        emoji_ratio        : rasio emoji / panjang teks
        uppercase_ratio    : rasio huruf kapital / panjang teks
        special_char_ratio : rasio karakter non-alphanumeric / panjang teks
        repeat_char_ratio  : rasio pola pengulangan (≥ 3 karakter berurutan)
    """
    if pd.isna(text):
        text = ""
    text = str(text)
    length = len(text)

    return {
        "text_length":        length,
        "word_count":         len(tokenize(text)),
        "emoji_ratio":        clip(safe_ratio(len(EMOJI_PATTERN.findall(text)), length)),
        "uppercase_ratio":    clip(safe_ratio(sum(c.isupper() for c in text), length)),
        "special_char_ratio": clip(safe_ratio(len(re.findall(r'[^a-zA-Z0-9\s]', text)), length)),
        "repeat_char_ratio":  clip(safe_ratio(len(re.findall(r'(.)\1{2,}', text)), length)),
    }


print("text_surface_features defined (fitur 1–6)")

text_surface_features defined (fitur 1–6)


### Cell 8 — Obfuscation Features (Fitur 7–12)

In [9]:
# ── Fitur 7: unicode_weird_ratio ──────────────────────────────────────────
def unicode_weird_ratio(text: str) -> float:
    """
    Rasio karakter Unicode dengan code point > 10000.
    Mendeteksi mathematical alphanumerics, enclosed chars, dll.
    """
    if not text:
        return 0.0
    weird = sum(1 for c in text if ord(c) > 10000)
    return safe_ratio(weird, len(text))


# ── Fitur 8: invisible_char_ratio ─────────────────────────────────────────
def invisible_char_ratio(text: str) -> float:
    """
    Rasio karakter kontrol (Cc) dan format (Cf) Unicode.
    Mendeteksi zero-width characters, BOM, BIDI marks, dll.
    """
    if not text:
        return 0.0
    invisible = sum(
        1 for c in text
        if unicodedata.category(c) in ("Cf", "Cc")
    )
    return safe_ratio(invisible, len(text))


# ── Fitur 9: masked_keyword_ratio ─────────────────────────────────────────
def masked_keyword_ratio(text: str) -> float:
    """
    Rasio pola alfabet yang diselingi karakter khusus.
    Contoh: 'S!L0T', 'G@C0R' → terdeteksi sebagai masked keyword.
    """
    if not text:
        return 0.0
    patterns = re.findall(r'[a-zA-Z][^a-zA-Z0-9\s]{1,3}[a-zA-Z]', text)
    return safe_ratio(len(patterns), len(text))


# ── Fitur 10: stretched_word_ratio ────────────────────────────────────────
def stretched_word_ratio(text: str) -> float:
    """
    Rasio kata dengan pengulangan karakter ≥ 4 kali berturut-turut.
    Contoh: 'gaaacooorrr', 'bossss' → stretched.
    (Lebih strict dari repeat_char_ratio yang mendeteksi ≥ 3)
    """
    if not text:
        return 0.0
    stretched = re.findall(r'(.)\1{3,}', text)
    return safe_ratio(len(stretched), len(text))


# ── Helper untuk slang features ───────────────────────────────────────────
# Leet-map khusus untuk normalisasi token saat cek slang
# (terpisah dari LEET_NORMALIZATION yang untuk full-text normalization)
_TOKEN_LEET_MAP = str.maketrans({
    '4': 'a', '@': 'a',
    '3': 'e',
    '1': 'i',
    '$': 's', '5': 's',
    '7': 't',
})

def _normalize_token_for_slang(token: str) -> str:
    """Normalisasi token untuk pencocokan kamus slang."""
    token = token.lower()
    token = token.translate(_TOKEN_LEET_MAP)
    token = re.sub(r'(.)\1{2,}', r'\1', token)
    return token


# ── Fitur 11: slang_ratio ─────────────────────────────────────────────────
def slang_ratio(text: str) -> float:
    """
    Rasio kata yang ada di kamus slang terhadap total kata.
    Mendeteksi penggunaan bahasa informal.
    """
    tokens = tokenize(text)
    if not tokens:
        return 0.0
    hits = sum(1 for t in tokens if t in SLANG_DICT)
    return hits / len(tokens)


# ── Fitur 12: slang_distortion_ratio ──────────────────────────────────────
def slang_distortion_ratio(text: str) -> float:
    """
    Rasio kata slang yang ditulis dalam bentuk distorsi (leet/stretched).
    Contoh: 'g4c0r' → normalized jadi 'gacor' → ada di SLANG_DICT → distorted.
    """
    tokens = tokenize(text)
    if not tokens:
        return 0.0

    distorted_hits = 0
    for t in tokens:
        norm_t = _normalize_token_for_slang(t)
        # Distorted = setelah normalisasi ada di slang dict, tapi aslinya berbeda
        if norm_t in SLANG_DICT and t != norm_t:
            distorted_hits += 1

    return distorted_hits / len(tokens)


print("Obfuscation features defined (fitur 7–12)")
print("  - unicode_weird_ratio")
print("  - invisible_char_ratio")
print("  - masked_keyword_ratio")
print("  - stretched_word_ratio")
print("  - slang_ratio")
print("  - slang_distortion_ratio")

Obfuscation features defined (fitur 7–12)
  - unicode_weird_ratio
  - invisible_char_ratio
  - masked_keyword_ratio
  - stretched_word_ratio
  - slang_ratio
  - slang_distortion_ratio


### Cell 9 — Master Feature Extraction Function

In [10]:
def extract_all_features(text) -> dict:
    """
    Ekstrak semua 12 fitur handcrafted dari teks.
    
    ⚠️ Harus dipanggil pada teks ASLI (sebelum normalisasi Unicode).
    """
    if pd.isna(text):
        text = ""
    text = str(text)

    surface = text_surface_features(text)

    obf = {
        "unicode_weird_ratio":    unicode_weird_ratio(text),
        "invisible_char_ratio":   invisible_char_ratio(text),
        "masked_keyword_ratio":   masked_keyword_ratio(text),
        "stretched_word_ratio":   stretched_word_ratio(text),
        "slang_ratio":            slang_ratio(text),
        "slang_distortion_ratio": slang_distortion_ratio(text),
    }

    return {**surface, **obf}


# Definisi final nama-nama fitur (urutan konsisten)
HANDCRAFTED_FEATURES = [
    # Surface (6)
    "text_length",
    "word_count",
    "emoji_ratio",
    "uppercase_ratio",
    "special_char_ratio",
    "repeat_char_ratio",
    # Obfuscation (6)
    "unicode_weird_ratio",
    "invisible_char_ratio",
    "masked_keyword_ratio",
    "stretched_word_ratio",
    "slang_ratio",
    "slang_distortion_ratio",
]

assert len(HANDCRAFTED_FEATURES) == 12, f"Harus 12 fitur, dapat {len(HANDCRAFTED_FEATURES)}"

# Quick sanity check
test_comment = "ayo d4ft4r d1 s1n1 g@c0r bossss 🎰🎰🎰"
result = extract_all_features(test_comment)
print("Sanity check pada 1 komentar spam:")
for k, v in result.items():
    print(f"  {k:<25} = {v}")

Sanity check pada 1 komentar spam:
  text_length               = 35
  word_count                = 7
  emoji_ratio               = 0.02857142857142857
  uppercase_ratio           = 0.0
  special_char_ratio        = 0.11428571428571428
  repeat_char_ratio         = 0.05714285714285714
  unicode_weird_ratio       = 0.08571428571428572
  invisible_char_ratio      = 0.0
  masked_keyword_ratio      = 0.02857142857142857
  stretched_word_ratio      = 0.02857142857142857
  slang_ratio               = 0.0
  slang_distortion_ratio    = 0.0


---
## Bagian 3 — Aplikasi ke Dataset

> ⚠️ **Urutan yang BENAR:**  
> 1. Ekstrak 12 fitur dari `comment_text` ASLI  
> 2. BARU buat kolom `comment_text_normalized`  
>
> Jangan balik urutan ini — normalisasi menghilangkan sinyal obfuskasi  
> yang justru ingin diukur oleh fitur-fitur di atas.

### Cell 10 — Step 1: Ekstrak Fitur dari Teks ASLI

In [11]:
from tqdm import tqdm
tqdm.pandas(desc="Extracting features")

print("Memproses semua split files...")
print("Urutan: ekstrak fitur dari teks ASLI → normalisasi → encode label\n")

for split_name, input_path in SPLIT_FILES.items():
    output_path = input_path.replace(".csv", "_features.csv")
    
    print(f"{'='*50}")
    print(f"[{split_name.upper()}] {input_path} → {output_path}")
    print(f"{'='*50}")
    
    df = pd.read_csv(input_path)
    print(f"  Loaded: {len(df):,} rows")
    
    # ── Step 1: Ekstrak 12 fitur dari teks ASLI (sebelum normalisasi) ──
    print("  Step 1: Ekstrak 12 text features dari teks asli...")
    feature_series = df[TEXT_COL].progress_apply(extract_all_features)
    feature_df = pd.DataFrame(feature_series.tolist())
    
    # Gabungkan — drop kolom lama dulu jika ada (re-run safety)
    df = pd.concat(
        [df.drop(columns=feature_df.columns, errors="ignore"), feature_df],
        axis=1
    )
    
    # ── Step 2: Normalisasi Unicode → kolom baru ──────────────────────
    print("  Step 2: Normalisasi Unicode → comment_text_normalized...")
    df["comment_text_normalized"] = df[TEXT_COL].progress_apply(normalize_unicode)
    
    # ── Step 3: Encode label string → numerik ─────────────────────────
    print("  Step 3: Encode label...")
    df["label"] = df[LABEL_COL].map(LABEL_MAP)
    unknown = df["label"].isna().sum()
    if unknown > 0:
        print(f"  ⚠️  {unknown} baris dengan label tidak dikenal!")
    else:
        print(f"  Label dist: SPAM={df['label'].sum():,}  NOT_SPAM={(df['label']==0).sum():,}")
    
    # ── Save ──────────────────────────────────────────────────────────
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"  ✅ Saved → {output_path}  shape={df.shape}\n")

print("\n✅ Semua split files selesai diproses!")
print("Output files:")
for split, path in SPLIT_FILES.items():
    out = path.replace(".csv", "_features.csv")
    print(f"  {out}")

Memproses semua split files...
Urutan: ekstrak fitur dari teks ASLI → normalisasi → encode label

[TRAIN] ./train.csv → ./train_features.csv
  Loaded: 55,012 rows
  Step 1: Ekstrak 12 text features dari teks asli...


Extracting features: 100%|██████████| 55012/55012 [00:02<00:00, 24423.23it/s]


  Step 2: Normalisasi Unicode → comment_text_normalized...


Extracting features: 100%|██████████| 55012/55012 [00:00<00:00, 212084.44it/s]


  Step 3: Encode label...
  Label dist: SPAM=3,320  NOT_SPAM=51,692
  ✅ Saved → ./train_features.csv  shape=(55012, 34)

[VAL] ./val.csv → ./val_features.csv
  Loaded: 6,876 rows
  Step 1: Ekstrak 12 text features dari teks asli...


...

  Step 3: Encode label...
  Label dist: SPAM=415  NOT_SPAM=6,461
  ✅ Saved → ./val_features.csv  shape=(6876, 34)

[TEST] ./test.csv → ./test_features.csv
  Loaded: 6,877 rows
  Step 1: Ekstrak 12 text features dari teks asli...


Extracting features: 100%|██████████| 6877/6877 [00:00<00:00, 19921.30it/s]


  Step 2: Normalisasi Unicode → comment_text_normalized...


Extracting features: 100%|██████████| 6877/6877 [00:00<00:00, 225853.71it/s]

  Step 3: Encode label...
  Label dist: SPAM=415  NOT_SPAM=6,462
  ✅ Saved → ./test_features.csv  shape=(6877, 34)


✅ Semua split files selesai diproses!
Output files:
  ./train_features.csv
  ./val_features.csv
  ./test_features.csv


### Cell 11 — Step 2: Normalisasi Unicode (SETELAH ekstraksi fitur)

### Cell 12 — Verifikasi Fitur

### Cell 13 — Cek Label Encoding

---
## Bagian 4 — Save Output

---
## ✅ Checklist — Output dari Notebook Ini

Setelah semua cell selesai dijalankan, akan ada 3 file baru:

| File | Isi |
|---|---|
| `train_features.csv` | 55,012 rows + 12 text features + `comment_text_normalized` + `label` |
| `val_features.csv` | 6,876 rows + kolom yang sama |
| `test_features.csv` | 6,877 rows + kolom yang sama |

**Kolom baru yang ditambahkan:**
- `comment_text_normalized` — input untuk IndoRoBERTa
- `label` — 1=SPAM, 0=NOT_SPAM (numeric)
- 12 text features: `text_length` ... `slang_distortion_ratio`

**Langkah selanjutnya:**  
Buka `spamshield_experiments.ipynb` → load ke-3 file `*_features.csv`